In [0]:
# Agent2: Data Cleaning
import pandas as pd
import pyspark.sql.functions as F

# Load dataset from bronze table
df = spark.sql("SELECT * FROM samplesuperstore.bronzedata.orders")
print("Bronze dataset rows:", df.count())
df.printSchema()

In [0]:
# Numeric columns median fill
numeric_cols = [c for (c, t) in df.dtypes if t in ("int", "double", "float")]
for col in numeric_cols:
    median_val = df.approxQuantile(col, [0.5], 0.01)[0]
    df = df.withColumn(col, F.when(F.col(col).isNull(), median_val).otherwise(F.col(col)))

# Categorical columns mode fill
categorical_cols = [c for (c, t) in df.dtypes if t == "string"]
for col in categorical_cols:
    mode_val = df.groupBy(col).count().orderBy(F.desc("count")).first()[0]
    df = df.withColumn(col, F.when(F.col(col).isNull(), mode_val).otherwise(F.col(col)))


In [0]:
# Drop duplicates
df = df.dropDuplicates()

# Flag negative profit orders
df = df.withColumn("Profit", F.when(F.col("Profit") < 0, 0).otherwise(F.col("Profit")))


In [0]:
# Rename columns for consistency
df = df.withColumnRenamed("Ship Mode", "ShipMode") \
       .withColumnRenamed("Postal Code", "PostalCode")


In [0]:
# Drop old silver table to avoid schema conflicts
spark.sql("DROP TABLE IF EXISTS samplesuperstore.silverdata.orders")

# Save cleaned dataset
df.write.option("overwriteSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("samplesuperstore.silverdata.orders")

print("Cleaned dataset saved to silver layer.")


In [0]:
%sql
USE CATALOG samplesuperstore;

CREATE OR REPLACE FUNCTION silverdata.run_data_cleaning(dataset STRING)
RETURNS STRING
LANGUAGE PYTHON
AS $$
def run_data_cleaning(dataset: str) -> str:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()

    df = spark.table(dataset)
    cleaned_df = df.filter(df['Profit'].isNotNull())

    return f"Cleaned dataset rows: {cleaned_df.count()}"
$$;


In [0]:
%sql
-- Step 1: Switch to the correct catalog
USE CATALOG samplesuperstore;

-- Step 2: Call the UC function with silverdata.orders table
SELECT silverdata.run_data_cleaning('silverdata.orders');

-- Step 3: Verify function registration (optional check)
SHOW FUNCTIONS IN silverdata;
